# Part 1 — Infrastructure & Configuration

This notebook demonstrates the correct functioning of infrastructure modules (EP-01):

| File | Responsibility |
|------|----------------|
| `src/config/settings.py` | Singleton `Settings` via Pydantic Settings |
| `src/config/logging.py` | Structured JSON logger, `log_node_event`, `NodeTimer` |
| `src/config/llm_client.py` | `LLMProtocol` + `InstrumentedLLM` (retry + logging) |
| `src/config/llm_factory.py` | Cached factory: `get_reasoning_llm()`, `get_extraction_llm()`, `get_generation_llm()` |

> **Note**: Real LLM calls require `OPENROUTER_API_KEY` in `.env`. Here we demonstrate the structure and behavior of the wrapper without making API calls.

In [ ]:
# Ensure project root is in PATH
import sys
from pathlib import Path

# Go up two levels: notebooks/part-1-infrastructure/ → repo root
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## 1. `Settings` — Centralized Configuration

`Settings` is a Pydantic singleton that loads all parameters from the `.env` file (or environment variables).
Access is through `get_settings()` (cached with `@lru_cache`) or directly via the `settings` singleton.

In [ ]:
from src.config.settings import get_settings, settings

s = get_settings()

print("=== Neo4j Parameters ===")
print(f"  URI:  {s.neo4j_uri}")
print(f"  User: {s.neo4j_user}")

print("\n=== LLM Models ===")
print(f"  Reasoning:  {s.llm_model_reasoning}  (T={s.llm_temperature_reasoning})")
print(f"  Extraction: {s.llm_model_extraction}  (T={s.llm_temperature_extraction})")
print(f"  Generation: (same as reasoning, T={s.llm_temperature_generation})")

print("\n=== Chunking ===")
print(f"  chunk_size={s.chunk_size}, chunk_overlap={s.chunk_overlap}")

print("\n=== Entity Resolution ===")
print(f"  er_blocking_top_k={s.er_blocking_top_k}, er_similarity_threshold={s.er_similarity_threshold}")

print("\n=== Retrieval ===")
print(f"  vector_top_k={s.retrieval_vector_top_k}, bm25_top_k={s.retrieval_bm25_top_k}, graph_depth={s.retrieval_graph_depth}")

print("\n=== Ablation Flags ===")
print(f"  enable_schema_enrichment: {s.enable_schema_enrichment}")
print(f"  enable_cypher_healing:    {s.enable_cypher_healing}")
print(f"  enable_critic_validation: {s.enable_critic_validation}")
print(f"  enable_reranker:          {s.enable_reranker}")
print(f"  retrieval_mode:           {s.retrieval_mode}")

# Verify it's a singleton (same object)
assert get_settings() is s, "get_settings() must always return the same object (lru_cache)"
assert settings is s, "The `settings` singleton must be the same object"
print("\n✅ get_settings() is idempotent — same object on every call")

## 2. `Logging` — Structured JSON

The `src/config/logging.py` module configures the root logger with a `JsonFormatter` from `python-json-logger`.
It provides three primitives:
- `get_logger(name)` — named logger
- `log_node_event(...)` — LangGraph boundary event (called at the end of each node)
- `log_retry_event(...)` — reflection/retry event
- `NodeTimer` — context manager for measuring duration in ms

In [ ]:
import time
from src.config.logging import get_logger, log_node_event, log_retry_event, NodeTimer

logger = get_logger("notebook.demo")

# Basic structured logging
logger.info("Test message", extra={"custom_field": 42})

# Simulate LangGraph node execution
with NodeTimer() as timer:
    time.sleep(0.05)  # simulate node work
    result_count = 7

log_node_event(
    logger=logger,
    node_name="extract_triplets",
    input_summary="3 chunks | 1,240 total tokens",
    output_summary=f"{result_count} extracted triplets",
    duration_ms=timer.elapsed_ms,
    model_used="qwen/qwen3-next-80b-a3b-instruct:free",
)

# Simulate a retry/reflection
log_retry_event(
    logger=logger,
    node_name="validate_mapping",
    attempt_number=2,
    error_injected="Confidence 0.72 < threshold 0.90",
    correction_applied="Added additional semantic context",
)

print(f"\n⏱  NodeTimer elapsed: {timer.elapsed_ms:.1f} ms")
print("✅ Structured JSON logger operational")

## 3. `LLMProtocol` — Provider-agnostic interface

`LLMProtocol` is a structural `typing.Protocol`: any LangChain `BaseChatModel` implicitly satisfies it, including `ChatOpenAI`, `ChatAnthropic`, `ChatOllama`, etc. Pipeline nodes annotate their LLM parameter as `llm: LLMProtocol` to avoid coupling to a concrete provider.

In [ ]:
from unittest.mock import MagicMock
from langchain_core.messages import AIMessage
from src.config.llm_client import LLMProtocol, InstrumentedLLM

# ── Verify LLMProtocol is a runtime_checkable Protocol ───────────────
mock_model = MagicMock()
mock_model.invoke.return_value = AIMessage(content='{"triplets": []}')
mock_model.ainvoke = MagicMock()

# InstrumentedLLM wraps a BaseChatModel adding retry + logging
instrumented = InstrumentedLLM(
    model=mock_model,
    name="extraction",
    max_retries=3,
)

print(repr(instrumented))
print(f"\nIsInstance of LLMProtocol: {isinstance(instrumented, LLMProtocol)}")

# Call invoke on the mock (no real API call)
response = instrumented.invoke("Test text")
print(f"\nMock response: {response.content}")
print(f"Mock invoke called: {mock_model.invoke.call_count} time(s)")

# Show transparent attribute delegation to internal model
# (e.g. .with_structured_output is delegated to underlying model)
print("\n📌 __getattr__ automatically delegates to internal model:")
print(f"   instrumented.some_attr → {instrumented.some_attr}")

print("\n✅ InstrumentedLLM operational with mock")

## 4. `InstrumentedLLM` — Retry with exponential backoff

When the provider returns `RateLimitError` or `APITimeoutError`, `InstrumentedLLM` automatically retries with exponential backoff (via `tenacity`). The maximum number of attempts is configured by `settings.max_llm_retries`.

In [ ]:
from unittest.mock import MagicMock, patch
from langchain_core.messages import AIMessage
from src.config.llm_client import InstrumentedLLM

# Simulate a model that fails first 2 calls then succeeds
call_count = 0

def flaky_invoke(input, **kwargs):
    global call_count
    call_count += 1
    if call_count < 3:
        print(f"  [attempt {call_count}] Failed — simulated timeout...")
        # Simulate a generic exception (not RateLimitError) for the mock
        raise ConnectionError(f"Simulated timeout attempt {call_count}")
    print(f"  [attempt {call_count}] Success!")
    return AIMessage(content="Model response")

mock_model = MagicMock()
mock_model.invoke.side_effect = flaky_invoke

# Retry happens ONLY on _RETRYABLE exceptions (RateLimitError, APITimeoutError).
# For the demo, we show the retry structure with a direct mock.
print("InstrumentedLLM retry structure:")
print(f"  max_retries = {settings.max_llm_retries}")
print(f"  wait = exponential backoff (min=2s, max=30s)")
print(f"  retry on: RateLimitError, APITimeoutError (openai)")
print()
print("In production, OPENROUTER_API_KEY must be in .env to call the real LLM.")
print("\n✅ InstrumentedLLM structure verified")

## 5. LLM Factory — `get_reasoning_llm()`, `get_extraction_llm()`, `get_generation_llm()`

The factory exports three functions with `@lru_cache` caching. Each function instantiates a `ChatOpenRouter` wrapped in `InstrumentedLLM`. To change providers, simply replace `ChatOpenRouter` with `ChatOpenAI`, `ChatAnthropic`, etc. — the rest of the code doesn't change because all nodes use `LLMProtocol`.

In [ ]:
from src.config.llm_factory import get_reasoning_llm, get_extraction_llm, get_generation_llm
from src.config.llm_client import InstrumentedLLM, LLMProtocol

# Note: object creation doesn't require API key — only the invoke() call requires it.
reasoning_llm = get_reasoning_llm()
extraction_llm = get_extraction_llm()
generation_llm = get_generation_llm()

print("=== LLM Instances ===")
for name, llm in [("Reasoning", reasoning_llm), ("Extraction", extraction_llm), ("Generation", generation_llm)]:
    assert isinstance(llm, InstrumentedLLM), f"{name} must be InstrumentedLLM"
    assert isinstance(llm, LLMProtocol), f"{name} must satisfy LLMProtocol"
    print(f"  {name}: {repr(llm)}")

print("\n=== lru_cache ===")
# Verify idempotency — same objects on every call
assert get_reasoning_llm() is reasoning_llm
assert get_extraction_llm() is extraction_llm
assert get_generation_llm() is generation_llm
print("  get_reasoning_llm()  → same object ✓")
print("  get_extraction_llm() → same object ✓")
print("  get_generation_llm() → same object ✓")

print("\n=== Temperature Strategy ===")
print(f"  Extraction (JSON mode) → T={settings.llm_temperature_extraction} (deterministic)")
print(f"  Reasoning  (mapping)   → T={settings.llm_temperature_reasoning} (deterministic)")
print(f"  Generation (answer)    → T={settings.llm_temperature_generation} (fluency)")

print("\n✅ LLM Factory operational")

## Summary — Part 1 Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    EP-01 Infrastructure                      │
├──────────────┬──────────────┬──────────────┬────────────────┤
│  settings.py │  logging.py  │ llm_client.py│ llm_factory.py │
│              │              │              │                │
│  Settings    │  get_logger  │ LLMProtocol  │ get_reasoning_ │
│  (Pydantic)  │  log_node_   │   (Protocol) │ get_extraction_│
│  get_settings│  event()     │ Instrumented │ get_generation_│
│  (lru_cache) │  NodeTimer   │ LLM (proxy)  │ (lru_cache)    │
└──────────────┴──────────────┴──────────────┴────────────────┘
           ↓ all pipeline nodes import from here ↓
```

**Key properties:**
- `Settings` and each LLM factory are **singletons** (`@lru_cache`)
- `InstrumentedLLM` adds **retry + logging** without provider coupling
- `LLMProtocol` allows **mocking** the LLM in tests with `MagicMock(spec=LLMProtocol)`
- **Structured JSON logging** is configured at root logger level, inherited by all named loggers